In [2]:
import re
from collections import defaultdict,Counter
import math
import random
import json
import pickle

In [8]:
telugu_sentences = [
    "రాజు పుస్తకం చదువుతున్నాడు",
    "సూర్యుడు ఉదయమవుతోంది",
    "పిల్లలు పార్క్ లో ఆడుతున్నారు",
    "రాము మరియు నందిని సముద్ర తీరంలో తిరుగుతున్నారు",
    "ఆమె వంటగదిలో కుక్కీ తయారు చేస్తోంది",
    "సైనికులు తమ విధులు నిబద్ధతగా చేస్తున్నారు",
    "బెంగళూరు లో కొత్త విద్యుత్ ప్రాజెక్ట్ ప్రారంభమైంది",
    "అతను నేటి వార్తల గురించి చదువుతున్నాడు",
    "వర్షం కారణంగా ఆట రద్దు అయ్యింది",
    "రాజు మరియు రాము బ్లాక్ లో ఆడుతున్నారు"
]

In [9]:
def tokenize_telugu_sentence(sentence):
    return [w for w in sentence.split() if re.match(r'[\u0C00-\u0C7F]+', w)]

tokenized_sentences = [tokenize_telugu_sentence(s) for s in telugu_sentences]

In [20]:
transition_counts = defaultdict(Counter)
word_counts = Counter()

for sent in tokenized_sentences:
    for i in range(len(sent) - 1):
        w1, w2 = sent[i], sent[i + 1]
        transition_counts[w1][w2] += 1
        word_counts[w1] += 1

In [15]:
transition_counts['రాజు'],transition_counts['రాము'],transition_counts['మరియు']

(Counter({'పుస్తకం': 1, 'మరియు': 1}),
 Counter({'మరియు': 1, 'బ్లాక్': 1}),
 Counter({'నందిని': 1, 'రాము': 1}))

In [16]:
word_counts['కుక్కీ']

1

In [21]:
#P(next_word|current_word)
transition_probs = defaultdict(dict)
vocab = {w for sent in tokenized_sentences for w in sent}  

for w1 in transition_counts:
    total = sum(transition_counts[w1].values())+len(vocab)    
    '''P(w2|w1)=count(w1->w2)/summation of w` count(w1->w`)
    for laplace smoothing we add 1 in the numerator and for denom len(vocab)
    '''
    for w2 in vocab:
        transition_probs[w1][w2]=(transition_counts[w1].get(w2, 0)+1)/total

In [22]:
def predict_next_word(context_word,top_k=5):
    if context_word not in transition_probs:
        return None
    probs=transition_probs[context_word]
    return sorted(probs.items(),key=lambda item:item[1],reverse=True)[:top_k]

In [ ]:
predict_next_word(context_word="రాజు")

[('మరియు', 0.047619047619047616),
 ('పుస్తకం', 0.047619047619047616),
 ('రద్దు', 0.023809523809523808),
 ('లో', 0.023809523809523808),
 ('అతను', 0.023809523809523808)]

In [ ]:
def complete_sentence(seed_word, max_len=10):
    sentence = [seed_word]
    current = seed_word
    for _ in range(max_len - 1):
        preds = predict_next_word(current, top_k=1)
        if not preds:
            break
        next_word = preds[0][0]
        if next_word in sentence:  
            break
        sentence.append(next_word)
        current = next_word
    return " ".join(sentence)

In [ ]:
complete_sentence('రాజు')

'రాజు మరియు నందిని సముద్ర తీరంలో తిరుగుతున్నారు'

In [ ]:

class BigramHMM:
    def __init__(self):
        self.transition_counts = defaultdict(Counter)
        self.word_counts = Counter()
        self.transition_probs = defaultdict(dict)
        self.vocab = set()
        self.trained = False
        self.vocab_size = 0

    def tokenize_telugu_sentence(self, sentence):
        return [w for w in sentence.split() if re.match(r'[\u0C00-\u0C7F]+', w)]

    def train(self, tokenized_sentences):
        self.transition_counts = defaultdict(Counter)
        self.word_counts = Counter()
        self.vocab = set()

        if tokenized_sentences and isinstance(tokenized_sentences[0], str):
            tokenized_sentences = [self.tokenize_telugu_sentence(s) for s in tokenized_sentences]

        for sent in tokenized_sentences:
            for i in range(len(sent) - 1):
                w1, w2 = sent[i], sent[i + 1]
                self.transition_counts[w1][w2] += 1
                self.word_counts[w1] += 1
                self.vocab.update([w1, w2])

        self.vocab_size = len(self.vocab)

        for w1 in self.transition_counts:
            total_transitions = sum(self.transition_counts[w1].values())
            denominator = total_transitions + self.vocab_size
            
            for w2, count in self.transition_counts[w1].items():
                self.transition_probs[w1][w2] = (count + 1) / denominator

        self.trained = True

    def predict_next(self, context, top_k=5):
        if isinstance(context, list):
            context_word = context[-1]
        else:
            context_word = context

        if context_word not in self.transition_probs:
            return []

        probs = self.transition_probs[context_word]
        sorted_predictions = sorted(probs.items(), key=lambda x: x[1], reverse=True)
        return sorted_predictions[:top_k]

    def complete_sentence(self, seed_word, max_len=10):
        sentence = [seed_word]
        current = seed_word
        for _ in range(max_len - 1):
            preds = self.predict_next(current, top_k=1)
            if not preds:
                break
            next_word = preds[0][0]
            if next_word in sentence:
                break
            sentence.append(next_word)
            current = next_word
        return " ".join(sentence)

    def get_probability(self, w1, w2):
        if w1 in self.transition_probs and w2 in self.transition_probs[w1]:
            return self.transition_probs[w1][w2]
        elif w1 in self.transition_counts:
            total_transitions = sum(self.transition_counts[w1].values())
            return 1 / (total_transitions + self.vocab_size)
        else:
            return 1 / self.vocab_size

    def perplexity(self, test_sentences, batch_size=1000):
        if test_sentences and isinstance(test_sentences[0], str):
            test_sentences = [self.tokenize_telugu_sentence(s) for s in test_sentences]

        total_log_prob = 0
        total_bigrams = 0

        for i in range(0, len(test_sentences), batch_size):
            batch = test_sentences[i:i + batch_size]
            for sent in batch:
                for j in range(len(sent) - 1):
                    w1, w2 = sent[j], sent[j + 1]
                    prob = self.get_probability(w1, w2)
                    total_log_prob += -math.log(prob)
                    total_bigrams += 1

        return math.exp(total_log_prob / total_bigrams) if total_bigrams > 0 else float('inf')

In [ ]:
model = BigramHMM()
model.train([
    "రాజు పుస్తకం చదువుతున్నాడు",
    "సూర్యుడు ఉదయమవుతోంది",
    "పిల్లలు పార్క్ లో ఆడుతున్నారు",
    "రాము మరియు నందిని సముద్ర తీరంలో తిరుగుతున్నారు",
    "ఆమె వంటగదిలో కుక్కీ తయారు చేస్తోంది",
    "సైనికులు తమ విధులు నిబద్ధతగా చేస్తున్నారు",
    "బెంగళూరు లో కొత్త విద్యుత్ ప్రాజెక్ట్ ప్రారంభమైంది",
    "అతను నేటి వార్తల గురించి చదువుతున్నాడు",
    "వర్షం కారణంగా ఆట రద్దు అయ్యింది",
    "రాజు మరియు రాము బ్లాక్ లో ఆడుతున్నారు"
])

print(model.predict_next("తీరంలో"))
sentence=model.complete_sentence("రాజు")
print(sentence)
print(model.perplexity(tokenized_sentences))


[('తిరుగుతున్నారు', 0.04878048780487805)]
రాజు పుస్తకం చదువుతున్నాడు
20.212016384110463


In [5]:
import math
from collections import defaultdict, Counter
import re

class BigramHMM:
    def __init__(self):
        self.transition_counts = defaultdict(Counter)
        self.word_counts = Counter()
        self.transition_probs = defaultdict(dict)
        self.vocab = set()
        self.trained = False
        self.vocab_size = 0

    def tokenize_telugu_sentence(self, sentence):
        return [w for w in sentence.split() if re.match(r'[\u0C00-\u0C7F]+', w)]

    def train(self, tokenized_sentences):
        self.transition_counts = defaultdict(Counter)
        self.word_counts = Counter()
        self.vocab = set()

        if tokenized_sentences and isinstance(tokenized_sentences[0], str):
            tokenized_sentences = [self.tokenize_telugu_sentence(s) for s in tokenized_sentences]

        for sent in tokenized_sentences:
            for i in range(len(sent) - 1):
                w1, w2 = sent[i], sent[i + 1]
                self.transition_counts[w1][w2] += 1
                self.word_counts[w1] += 1
                self.vocab.update([w1, w2])

        self.vocab_size = len(self.vocab)

        for w1 in self.transition_counts:
            total_transitions = sum(self.transition_counts[w1].values())
            denominator = total_transitions + self.vocab_size
            
            for w2, count in self.transition_counts[w1].items():
                self.transition_probs[w1][w2] = (count + 1) / denominator

        self.trained = True

    def predict_next(self, context, top_k=5):
        if isinstance(context, list):
            context_word = context[-1]
        else:
            context_word = context

        if context_word not in self.transition_probs:
            return []

        probs = self.transition_probs[context_word]
        sorted_predictions = sorted(probs.items(), key=lambda x: x[1], reverse=True)
        return sorted_predictions[:top_k]

    def complete_sentence(self, seed_word, max_len=40):
        sentence = [seed_word]
        current = seed_word
        for _ in range(max_len - 1):
            preds = self.predict_next(current, top_k=1)
            if not preds:
                break
            next_word = preds[0][0]
            if next_word in sentence:
                break
            sentence.append(next_word)
            current = next_word
        return " ".join(sentence)

    def get_probability(self, w1, w2):
        if w1 in self.transition_probs and w2 in self.transition_probs[w1]:
            return self.transition_probs[w1][w2]
        elif w1 in self.transition_counts:
            total_transitions = sum(self.transition_counts[w1].values())
            return 1 / (total_transitions + self.vocab_size)
        else:
            return 1 / self.vocab_size

    def perplexity(self, test_sentences, batch_size=1000):
        if test_sentences and isinstance(test_sentences[0], str):
            test_sentences = [self.tokenize_telugu_sentence(s) for s in test_sentences]

        total_log_prob = 0
        total_bigrams = 0

        for i in range(0, len(test_sentences), batch_size):
            batch = test_sentences[i:i + batch_size]
            for sent in batch:
                for j in range(len(sent) - 1):
                    w1, w2 = sent[j], sent[j + 1]
                    prob = self.get_probability(w1, w2)
                    total_log_prob += -math.log(prob)
                    total_bigrams += 1

        return math.exp(total_log_prob / total_bigrams) if total_bigrams > 0 else float('inf')

In [14]:
def load_telugu_data():
    try:
        with open('final_cleaned_telugu_data_1.txt', 'r', encoding='utf-8') as f:
            sentences = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(sentences)} sentences from final_cleaned_telugu_data.txt")
    except FileNotFoundError:
        print("final_cleaned_telugu_data.txt not found. Using sample sentences instead.")
        sentences = [
            "రాజు పుస్తకం చదువుతున్నాడు",
            "సూర్యుడు ఉదయమవుతోంది",
            "పిల్లలు పార్క్ లో ఆడుతున్నారు",
            "రాము మరియు నందిని సముద్ర తీరంలో తిరుగుతున్నారు",
            "ఆమె వంటగదిలో కుక్కీ తయారు చేస్తోంది"
        ]

    
    tokenized_sentences = [tokenize_telugu_sentence(s) for s in sentences]
    print(f"Tokenized {len(tokenized_sentences)} sentences")

    return tokenized_sentences


In [15]:
model=BigramHMM()
data=load_telugu_data()
model.train(data)
model.complete_sentence('రాజు')

Loaded 159164 sentences from final_cleaned_telugu_data.txt
Tokenized 159164 sentences


'రాజు తన మొదటి ప్రపంచ యుద్ధం తరువాత లో జరిగిన మ్యాచ్లో టాస్ గెలిచి ఫీల్డింగ్ ఎంచుకుంది.'

## HMM + NGram Integration

In [2]:
import re
from collections import defaultdict, Counter
import math

class HMMNGramModel:
    def __init__(self, n=3):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        self.vocab = set()
        self.trained = False
        
    def tokenize_telugu_sentence(self, sentence):
        return [w for w in sentence.split() if re.match(r'[\u0C00-\u0C7F]+', w)]

    def train(self, sentences,vocab=None):
        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()
        if vocab:
            self.vocab=vocab.copy()
        else:
            self.vocab=set()

        
        if sentences and isinstance(sentences[0], str):
            tokenized_sentences = [self.tokenize_telugu_sentence(s) for s in sentences]
        else:
            tokenized_sentences = sentences

        for tokens in tokenized_sentences:
            
            tokens = ["<s>"] * (self.n - 1) + tokens + ["</s>"]
            
            for i in range(len(tokens) - self.n + 1):
                context = tuple(tokens[i:i+self.n-1])
                next_word = tokens[i+self.n-1]
                if not vocab:
                    self.vocab.add(next_word)
                
                self.ngram_counts[context][next_word] += 1
                self.context_counts[context] += 1
        
        
        self.vocab.update(["<s>", "</s>"])
        self.trained = True

    def get_probability(self, context, word):
        
        context_size = self.n - 1
        current_context = tuple(context[-context_size:]) if len(context) >= context_size else tuple(["<s>"] * (context_size - len(context)) + context)
        
        vocab_size = len(self.vocab)
        count_word = self.ngram_counts.get(current_context, {}).get(word, 0)
        count_context = self.context_counts.get(current_context, 0)
        
        denominator = count_context + vocab_size
        
        if denominator > 0:
            return (count_word + 1) / denominator
        else:
            
            return 1 / vocab_size

    def predict_next(self, context, top_k=3):
        if isinstance(context, str):
            context = self.tokenize_telugu_sentence(context)
            
        context_size = self.n - 1
        current_context = tuple(["<s>"] * max(0, context_size - len(context)) + context[-context_size:])
        
       
        if current_context in self.context_counts:
            probs = {}
            for word in self.vocab:
                 if word not in ['<s>']:
                    
                    probs[word] = self.get_probability(current_context, word)
            
            predictions = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:top_k]
            return predictions
        
        
        if self.n > 2 and len(current_context) > 1:
            return self.predict_next(context[-1], top_k)
        
        
        word_freq = Counter()
        for cxt in self.context_counts:
            word_freq.update(self.ngram_counts[cxt])
            
        total = sum(word_freq.values())
        predictions = [(word, count/total) for word, count in word_freq.most_common(top_k) if word not in ['<s>', '</s>']]
        
        return predictions if predictions else [('చదువుతున్నాడు', 1 / len(self.vocab))]

    def complete_sentence(self, seed_words, max_len=20):
        
        if isinstance(seed_words, str):
            sentence = self.tokenize_telugu_sentence(seed_words)
        else:
            sentence = seed_words.copy()
            
        generated_words = set(sentence)
        
        for _ in range(max_len):
            next_word_pred = self.predict_next(sentence, top_k=1)
            
            if not next_word_pred:
                break
                
            next_word = next_word_pred[0][0]
            
          
            if next_word == '</s>':
                break
            
            
            if next_word in generated_words:
                is_short_word = len(next_word) <= 2
                
                if not is_short_word:
                    break 
                elif sentence.count(next_word) >= 2:
                    break 
                
            sentence.append(next_word)
            generated_words.add(next_word)
            
        return " ".join(sentence)

    def perplexity(self, test_sentences):
        
        if not self.trained:
            return float('inf')
            
        total_log_prob = 0
        total_words = 0
        
        for sentence in test_sentences:
            if isinstance(sentence, str):
                tokens = self.tokenize_telugu_sentence(sentence)
            else:
                tokens = sentence
                
            tokens = ["<s>"] * (self.n - 1) + tokens + ["</s>"]
            
            for i in range(self.n - 1, len(tokens)):
                context = tokens[i-self.n+1:i]
                word = tokens[i]
                
                
                prob = self.get_probability(context, word)
                total_log_prob += -math.log(max(prob, 1e-10))
                total_words += 1

        return math.log(total_log_prob / total_words) if total_words > 0 else float('inf')

In [ ]:
telugu_sentences = [
    "రాజు పుస్తకం చదువుతున్నాడు",
    "సూర్యుడు ఉదయమవుతోంది",
    "పిల్లలు పార్క్ లో ఆడుతున్నారు",
    "రాము మరియు నందిని సముద్ర తీరంలో తిరుగుతున్నారు",
    "ఆమె వంటగదిలో కుక్కీ తయారు చేస్తోంది",
    "సైనికులు తమ విధులు నిబద్ధతగా చేస్తున్నారు",
    "బెంగళూరు లో కొత్త విద్యుత్ ప్రాజెక్ట్ ప్రారంభమైంది",
    "అతను నేటి వార్తల గురించి చదువుతున్నాడు",
    "వర్షం కారణంగా ఆట రద్దు అయ్యింది",
    "రాజు మరియు రాము బ్లాక్ లో ఆడుతున్నారు"
]


In [ ]:
def tokenize_telugu_sentence(sentence):
    return [w for w in sentence.split() if re.match(r'[\u0C00-\u0C7F]+', w)]

tokenized_sentences = [tokenize_telugu_sentence(s) for s in telugu_sentences]

model = HMMNGramModel(n=3)
model.train(tokenized_sentences)

In [ ]:
phrases = [
    ["రాజు"],
    ["పిల్లలు", "పార్క్"],
    ["రాము", "మరియు"],
    ["ఆమె", "వంటగదిలో"],
    ["బెంగళూరు", "లో"]
]

for phrase in phrases:
    preds = model.predict_next(phrase)
    print(f"Input: {' '.join(phrase)}")
    if preds:
        for word, p in preds:
            print(f"  {word}: {p:.3f}")
    else:
        print("  No prediction found")
    print()

Input: రాజు
  మరియు: 0.045
  పుస్తకం: 0.045
  బ్లాక్: 0.023

Input: పిల్లలు పార్క్
  లో: 0.047
  బ్లాక్: 0.023
  విధులు: 0.023

Input: రాము మరియు
  నందిని: 0.047
  బ్లాక్: 0.023
  విధులు: 0.023

Input: ఆమె వంటగదిలో
  కుక్కీ: 0.047
  బ్లాక్: 0.023
  విధులు: 0.023

Input: బెంగళూరు లో
  కొత్త: 0.047
  బ్లాక్: 0.023
  విధులు: 0.023



In [ ]:
print("Sentence Completion: ")
completed = model.complete_sentence("రాజు")
print(f"Completed: {completed}")


Sentence Completion: 
Completed: రాజు మరియు రాము బ్లాక్ లో ఆడుతున్నారు


In [ ]:
data=load_telugu_data()

Loaded 179132 sentences from final_cleaned_telugu_data.txt
Tokenized 179132 sentences


In [ ]:
model=HMMNGramModel(n=3)
model.train(data)
model.perplexity(data)

2.4279647096248245

In [ ]:
model.complete_sentence('బౌద్ధ')

'బౌద్ధ జాతక కథలలో ఆంధ్రాపధం భీమసేన జాతకం, ఆంధ్రనగరి సెరివణిజ జాతకం ప్రస్తావన ఉంది.'

# Final Test on Large Dataset

In [3]:
with open('telugu_tokenized_sentences.json','r',encoding='utf-8') as f:
    tokenized_sentences=json.load(f)
    
with open('telugu_vocabulary.txt','r',encoding='utf-8') as f:
    vocab={line.strip() for line in f if line.strip()}


print(f"Loaded {len(tokenized_sentences)} tokenized sentences and {len(vocab)} vocab words.")

Loaded 1036856 tokenized sentences and 959816 vocab words.


In [4]:
model = HMMNGramModel(n=3)
model.train(tokenized_sentences,vocab=vocab)

In [17]:
print("Perplexity:", model.perplexity(tokenized_sentences))

Perplexity: 2.532429908274099


In [12]:
print(list(model.vocab)[:10])

['పట్టుకోబోతూ', 'ద్రవ్యాలతయారీకి', 'వెన్నునాడుల', 'వాజపేయిల్లో', 'వెళ్ళేటైంకీ', 'వెస్టిండీ్సలో', 'హేబలపుత్రా', 'ఆగిపోయిందేమో', 'ఈయనకి', 'చంపబోయారు']


In [16]:
word="పన్నెండున్నరకి"
print("Prediction:", model.predict_next([word]))
print("Completion:", model.complete_sentence(word))

Prediction: [('తన', 2.0836371970912424e-06), ('పేల్చారు', 1.0418185985456212e-06), ('ఇలాంటిప్రశ్నలు', 1.0418185985456212e-06)]
Completion: పన్నెండున్నరకి తన ఫ్లాట్ కి తాళం వేసి వుంది


In [19]:
word="సూర్య"
print("Prediction:", model.predict_next([word]))
print("Completion:", model.complete_sentence(word))

Prediction: [('చంద్ర', 6.250279960456562e-06), ('చంద్రులు', 4.166853306971042e-06), ('భరత్', 4.166853306971042e-06)]
Completion: సూర్య చంద్ర కు భూలోకమ్మ చెప్పిన మాటలు గుర్తొచ్చాయి


In [25]:
word="చదువు"
print("Prediction:", model.predict_next([word]))
print("Completion:", model.complete_sentence(word))

Prediction: [('పూర్తి', 6.249934896511495e-06), ('సంధ్యలు', 6.249934896511495e-06), ('మానేసి', 5.208279080426246e-06)]
Completion: చదువు పూర్తి అయింది


In [6]:
word="రాము"
print("Prediction:", model.predict_next([word]))
print("Completion:", model.complete_sentence(word))

Prediction: [('పెద్దగా', 3.1254036979776555e-06), ('వచ్చాడేమో', 2.083602465318437e-06), ('తిరిగి', 2.083602465318437e-06)]
Completion: రాము పెద్దగా నవ్వుకుంటూ తుర్రున బయటకు పరుగెత్తింది


In [7]:
word="సైనికులు"
print("Prediction:", model.predict_next([word]))
print("Completion:", model.complete_sentence(word))

Prediction: [('క్రమశిక్షణకి', 2.0836002946210817e-06), ('పంటితో', 2.0836002946210817e-06), ('కవాతు', 2.0836002946210817e-06)]
Completion: సైనికులు క్రమశిక్షణకి అలవాటుపడినవారు కాబట్టి తమ కోర్కెని పై అధికారులకి చెప్పవచ్చు


In [ ]:
word="వాణిజ్యం"
print("Prediction:", model.predict_next([word]))                        #Commerce allowed culture to flourish.
print("Completion:", model.complete_sentence(word))

Prediction: [('సంస్కృతి', 2.083628514039489e-06), ('మూలధన', 2.083628514039489e-06), ('మలయా', 2.083628514039489e-06)]
Completion: వాణిజ్యం సంస్కృతి వృద్ధి చెందేందుకు అనుమతించింది


In [ ]:
word="ఇల్లు"
print("Prediction:", model.predict_next([word]))                        #He sold his house and paid his debts
print("Completion:", model.complete_sentence(word))

Prediction: [('కూడా', 9.373437760373271e-06), ('చాలా', 8.331944675887353e-06), ('ఖాళీ', 6.248958506915514e-06)]
Completion: ఇల్లు కూడా అమ్మి అప్పులన్నీ తీర్చేశాడు


In [17]:
word="మాట వింటూనే"
print("Prediction:", model.predict_next([word]))                   
print("Completion:", model.complete_sentence(word))

Prediction: [('ఆమె', 3.1253483461177446e-06), ('కూర్మావతి', 2.083565564078496e-06), ('అపొజిషన్', 2.083565564078496e-06)]
Completion: మాట వింటూనే ఆమె తల్లిదండ్రులిద్దరూ విరుచుకుపడి పోయారు


In [19]:
word="పరిశుభ్రత"
print("Prediction:", model.predict_next([word]))                   
print("Completion:", model.complete_sentence(word))

Prediction: [('విషయంలో', 2.0836328555563195e-06), ('ప్రజారోగ్యం', 2.0836328555563195e-06), ('కోసం', 2.0836328555563195e-06)]
Completion: పరిశుభ్రత విషయంలో శ్రద్ధ తీసుకోవడం లేదేమోనని భయం కలిగింది


In [ ]:
word="ఆమె"
print("Prediction:", model.predict_next([word]))                   
print("Completion:", model.complete_sentence(word))